# 16

In [47]:
import pandas as pd
from pathlib import Path
import os

current_dir = Path.cwd()
data_path = os.path.join(current_dir, "Industrial_and_Scientific.json")

print("current_dir:", current_dir)
print("data_path:", data_path)

current_dir: C:\Users\leiqa\OneDrive\Desktop\Semester_06\NLP\W25_COMP262_002_TeamGamma
data_path: C:\Users\leiqa\OneDrive\Desktop\Semester_06\NLP\W25_COMP262_002_TeamGamma\Industrial_and_Scientific.json


In [48]:
#load data
dataset = pd.read_json(data_path,orient='records',lines=True)
#dataset = pd.read_csv(data_path)

In [49]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1758333 entries, 0 to 1758332
Data columns (total 12 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   overall         int64 
 1   verified        bool  
 2   reviewTime      object
 3   reviewerID      object
 4   asin            object
 5   reviewerName    object
 6   reviewText      object
 7   summary         object
 8   unixReviewTime  int64 
 9   vote            object
 10  style           object
 11  image           object
dtypes: bool(1), int64(2), object(9)
memory usage: 149.2+ MB


In [50]:
dataset.head(5)

,overall,verified,reviewTime,reviewerID,asin,reviewerName,reviewText,summary,unixReviewTime,vote,style,image
0,5,True,"01 23, 2013",A3FANY5GOT5X0W,0176496920,Kelly Keyser,"Arrived on time, in mint condition, great! I ...",Just as described!,1358899200,NaN,NaN,NaN
1,5,True,"11 5, 2012",AT6HRPPYOPHMB,0176496920,Michael C,This device was hard to find for my daughter's...,Great device,1352073600,NaN,NaN,NaN
2,4,True,"10 17, 2012",A4IX7B38LIN1E,0176496920,BH,Just a clicker nothing special. Was hoping it ...,Pretty Good,1350432000,NaN,NaN,NaN
3,5,True,"03 29, 2017",A12Q4LR8N17AOZ,0176496920,Waterfall3500,Great response card. Slow shipping but it work...,Thank you for the great product. Works. A++ Us...,1490745600,NaN,NaN,NaN
4,1,True,"03 21, 2017",A1GJXZZPOZ3OD9,0176496920,Amazon Customer,It only lasted for 3 days before it stopped wo...,One Star,1490054400,NaN,NaN,NaN


In [51]:
# Count words in each reviewText
dataset['word_count'] = dataset['reviewText'].apply(lambda x: len(str(x).split()))

long_reviews = dataset[dataset['word_count'] > 100].copy()
print(dataset['word_count'].describe())
print(dataset.sort_values('word_count', ascending=False).head(5)[['reviewText', 'word_count']])

selected_reviews = long_reviews.head(10)

print(selected_reviews[['reviewText']].iloc[:2])



count    1.758333e+06
mean     3.427038e+01
std      6.059225e+01
min      0.000000e+00
25%      6.000000e+00
50%      1.700000e+01
75%      3.900000e+01
max      5.966000e+03
Name: word_count, dtype: float64
                                               reviewText  word_count
236852  FOR OPERATING A REFRIGERATOR (or freezer ) ONL...        5966
279835  FOR OPERATING A REFRIGERATOR (or freezer ) ONL...        5966
573451  I think someone left their Super Nintendo in t...        5875
987865  EDIT NEW REVIEW FIRST NOW:\nDO NOT BUY THIS PR...        4150
77717   Amazon may be lumping reviews of CPU-1 and CPU...        3853
                                            reviewText
23   The device came fully charged, I would recomme...
127  This product has been around since i can remem...


In [52]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [53]:
import tensorflow as tf
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)

In [56]:
from transformers import pipeline
#summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
summarizer = pipeline("summarization", model="t5-small")

# Apply summarizer to each review
selected_reviews['summary'] = selected_reviews['reviewText'].apply(
    lambda x: summarizer(x, max_length=50, min_length=25, do_sample=False)[0]['summary_text']
)

print("Number of selected reviews:", selected_reviews.shape[0])

print("Summary 1:", selected_reviews.iloc[0]['summary'])
print("Summary 2:", selected_reviews.iloc[1]['summary'])

Device set to use cpu
Token indices sequence length is longer than the specified maximum sequence length for this model (513 > 512). Running this sequence through the model will result in indexing errors


Number of selected reviews: 10
Summary 1: the device came fully charged, I would recommend getting it used . it comes with a fully charged battery and is like new . there is also an app called ResponseWare that you can download on your smartphone and set up to work
Summary 2: this product has been around since i can remember . they are hard to find and when you do they are very expensive . great to see amazon has a company that sales them on this site .


C:\Users\leiqa\AppData\Local\Temp\ipykernel_28240\3221328418.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_reviews['summary'] = selected_reviews['reviewText'].apply(


In [58]:
# Filter for reviews that contain a question
question_reviews = dataset[dataset['reviewText'].str.contains(r'\?', na=False)]

# Select the first one or review more for a better one
question_review = question_reviews.iloc[0]
print("Question Review:\n", question_review['reviewText'])

Question Review:
 It should not even get a star!! It doesn't work and they won't take it back if you have pulled the battery tab...... how else would you discover it doesn't work if you don't pull the tab?! It would not register my response in class


In [59]:
"""from transformers import BlenderbotTokenizer, BlenderbotForConditionalGeneration

model_name = "facebook/blenderbot-400M-distill"
tokenizer = BlenderbotTokenizer.from_pretrained(model_name)
model = BlenderbotForConditionalGeneration.from_pretrained(model_name)

# Input
input_text = "Can this product be used with a MacBook Pro?"
inputs = tokenizer(input_text, return_tensors="pt")

# Generate reply
reply_ids = model.generate(**inputs, max_length=100)
response = tokenizer.decode(reply_ids[0], skip_special_tokens=True)

print("AI Service Rep Response:\n", response)"""

AI Service Rep Response:
  I'm not sure, but I'm sure it can. It's a very popular product.


In [61]:
chatbot = pipeline("text2text-generation", model="facebook/blenderbot-400M-distill")

input_text = "Can this product be used with a MacBook Pro?"
response = chatbot(input_text, max_length=100, do_sample=False)

print("🛎️ AI Service Rep Response:\n", response[0]['generated_text'])

Device set to use cpu


🛎️ AI Service Rep Response:
  I'm not sure, but I'm sure it can. It's a very popular product.
